# 06 — Advanced Features: Star Availability + Strength of Schedule

**Diagnose aus Notebook 05:** Team-aggregierte Box-Score-Features brachten nur +0.4pp Accuracy — im Rauschen. Grund: sie korrelieren stark mit Form & ELO, die wir schon haben.

**Hypothese:** Drei Konzepte tragen *neue* Information, die ELO/Form nicht enthalten:
1. **Star-Verfuegbarkeit** — wie viele der Kern-Spieler waren zuletzt im Lineup? Verletzungen = Wechsel der Team-Identitaet.
2. **Strength of Schedule** — eine 8-Sieg-Serie gegen Tank-Teams bedeutet etwas anderes als gegen Conference-Leader.
3. **Quality Wins** — Bilanz gegen Top-Gegner, gewichtet nach deren ELO.

**Goldene Regel** wie immer: kein Feature darf Information aus dem aktuellen Spiel enthalten.

## 1. Setup & Daten

In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.metrics import accuracy_score, log_loss, brier_score_loss, roc_auc_score
import xgboost as xgb

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)

DATA_RAW = Path('..') / 'data' / 'raw'
DATA_PROCESSED = Path('..') / 'data' / 'processed'

games = pd.read_parquet(DATA_PROCESSED / 'games_with_player_features.parquet')
print(f'Games: {len(games):,}  ({games.season.min()} - {games.season.max()})')

Games: 71,239  (1946 - 2025)


## 2. Star Availability — wer war wirklich im Lineup?

**Idee:** Pro Team und Saison identifizieren wir die **Top-5 Spieler nach Minuten** *bis zum aktuellen Spiel*. Dann zaehlen wir, wieviele davon in den letzten 3 Spielen aktiv waren.

Wenig Top-5 verfuegbar = Verletzungswelle = wahrscheinlich schlechtere Performance.

*Vereinfachung: Wir nutzen Saison-Top-5 statt rollierend Top-5 — das macht die Berechnung effizient ueber 1.6M Player-Game-Zeilen, ohne grosse Aenderung im Signal.*

In [7]:
ps = pd.read_csv(DATA_RAW / 'PlayerStatistics.csv',
                 usecols=['gameId', 'playerteamId', 'personId', 'numMinutes', 'comment'],
                 low_memory=False)
ps['played'] = (ps['numMinutes'].fillna(0) > 0).astype(int)

# Game-Datum + Saison anhaengen
ps = ps.merge(games[['gameId', 'gameDate', 'season']], on='gameId', how='inner')
print(f'Player-Game-Zeilen: {len(ps):,}')

TypeError: '>' not supported between instances of 'str' and 'int'

In [ ]:
# Saison-Top-5 pro Team: Spieler mit den meisten Saison-Minuten
season_minutes = ps.groupby(['season', 'playerteamId', 'personId'], as_index=False).agg(
    season_min=('numMinutes', 'sum')
)
season_minutes['rank'] = season_minutes.groupby(['season', 'playerteamId'])['season_min'].rank(method='dense', ascending=False)
top5 = season_minutes[season_minutes['rank'] <= 5][['season', 'playerteamId', 'personId']]
top5['is_top5'] = 1

ps = ps.merge(top5, on=['season', 'playerteamId', 'personId'], how='left')
ps['is_top5'] = ps['is_top5'].fillna(0).astype(int)
print(f'Top-5-Markierungen gesetzt: {ps.is_top5.sum():,}')

In [ ]:
# Pro Team-Game: wieviele der Top-5 haben gespielt?
core_avail = (
    ps[ps.is_top5 == 1]
    .groupby(['gameId', 'playerteamId'], as_index=False)
    .agg(top5_played=('played', 'sum'))
)
core_avail = core_avail.merge(games[['gameId', 'gameDate']], on='gameId').sort_values(['playerteamId', 'gameDate'])

# Rolling-Mittel der Top-5-Verfuegbarkeit ueber die letzten 3 Spiele (geshiftet — kein Leak)
core_avail['top5_avail_last3'] = (
    core_avail.groupby('playerteamId')['top5_played']
    .transform(lambda s: s.shift(1).rolling(3, min_periods=1).mean())
)
core_avail.head(8)

## 3. Strength of Schedule (SoS)

Wir haben pro Spiel das ELO beider Teams. Pro Team-Game-Sicht: das ELO des Gegners zum Spielzeitpunkt = Schwierigkeit.  
Rollender Schnitt der **letzten 10 Gegner-ELOs** = SoS der jueengsten Phase.

**Form korrigiert um SoS:** durchschnittliche Punktedifferenz minus durchschnittliches Gegner-ELO-Minus-Liga-Schnitt (~1500). Ein +5-Margin gegen Top-Teams ist mehr wert als +5 gegen Bottom-Teams.

In [ ]:
# Game -> Team-Game-View mit Gegner-ELO
tg_home = games[['gameId', 'gameDate', 'season', 'hometeamId', 'awayteamId',
                 'home_elo_pre', 'away_elo_pre', 'homeScore', 'awayScore']].copy()
tg_home = tg_home.rename(columns={
    'hometeamId': 'teamId', 'awayteamId': 'oppId',
    'home_elo_pre': 'team_elo', 'away_elo_pre': 'opp_elo',
    'homeScore': 'pts', 'awayScore': 'opp_pts',
})

tg_away = games[['gameId', 'gameDate', 'season', 'awayteamId', 'hometeamId',
                 'away_elo_pre', 'home_elo_pre', 'awayScore', 'homeScore']].copy()
tg_away = tg_away.rename(columns={
    'awayteamId': 'teamId', 'hometeamId': 'oppId',
    'away_elo_pre': 'team_elo', 'home_elo_pre': 'opp_elo',
    'awayScore': 'pts', 'homeScore': 'opp_pts',
})

tg = pd.concat([tg_home, tg_away], ignore_index=True).sort_values(['teamId', 'gameDate']).reset_index(drop=True)
tg['margin'] = tg.pts - tg.opp_pts
tg['win'] = (tg.margin > 0).astype(int)
tg['quality_win'] = ((tg.win == 1) & (tg.opp_elo > 1500)).astype(int)

grp = tg.groupby(['teamId', 'season'])
tg['sos_last10'] = grp['opp_elo'].transform(lambda s: s.shift(1).rolling(10, min_periods=3).mean())
tg['quality_win_rate_last10'] = grp['quality_win'].transform(lambda s: s.shift(1).rolling(10, min_periods=3).mean())
tg['sos_adj_margin_last10'] = grp.apply(
    lambda d: (d.margin - (1500 - d.opp_elo) / 25).shift(1).rolling(10, min_periods=3).mean(),
    include_groups=False,
).reset_index(level=[0, 1], drop=True)

sos_features = tg[['gameId', 'teamId', 'sos_last10', 'quality_win_rate_last10', 'sos_adj_margin_last10']]
sos_features.head(5)

## 4. Alle neuen Features zurueck auf Game-Level mergen

In [ ]:
# Star-Availability
g = games.merge(
    core_avail[['gameId', 'playerteamId', 'top5_avail_last3']]
        .rename(columns={'playerteamId': 'hometeamId', 'top5_avail_last3': 'home_top5_avail_last3'}),
    on=['gameId', 'hometeamId'], how='left'
)
g = g.merge(
    core_avail[['gameId', 'playerteamId', 'top5_avail_last3']]
        .rename(columns={'playerteamId': 'awayteamId', 'top5_avail_last3': 'away_top5_avail_last3'}),
    on=['gameId', 'awayteamId'], how='left'
)
g['top5_avail_diff'] = g.home_top5_avail_last3 - g.away_top5_avail_last3

# SoS
sos_cols = ['sos_last10', 'quality_win_rate_last10', 'sos_adj_margin_last10']
g = g.merge(
    sos_features.rename(columns={c: f'home_{c}' for c in sos_cols} | {'teamId': 'hometeamId'}),
    on=['gameId', 'hometeamId'], how='left'
)
g = g.merge(
    sos_features.rename(columns={c: f'away_{c}' for c in sos_cols} | {'teamId': 'awayteamId'}),
    on=['gameId', 'awayteamId'], how='left'
)
for c in sos_cols:
    g[f'{c}_diff'] = g[f'home_{c}'] - g[f'away_{c}']

g.to_parquet(DATA_PROCESSED / 'games_with_advanced_features.parquet', index=False)
print(f'Spalten gesamt: {g.shape[1]}')

## 5. Sanity-Check: tragen die neuen Features Signal?

Wenn `top5_avail_diff` > 0 (Heimteam hat mehr seiner Stars zuletzt gespielt), sollte die Heimsieg-Quote hoeher sein.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

tmp = g.dropna(subset=['top5_avail_diff', 'home_win'])
tmp['avail_bin'] = pd.cut(tmp.top5_avail_diff, bins=10)
tmp.groupby('avail_bin', observed=True).home_win.mean().plot(ax=axes[0], marker='o', color='steelblue')
axes[0].axhline(0.5, color='gray', linestyle='--')
axes[0].set_title('Heimsieg-Rate je Top-5-Verfuegbarkeits-Diff')
axes[0].set_xlabel('Diff (Heim - Auswaerts)')
axes[0].set_ylabel('Heimsieg-Rate')
axes[0].tick_params(axis='x', rotation=45)

tmp = g.dropna(subset=['sos_adj_margin_last10_diff', 'home_win'])
tmp['sos_bin'] = pd.cut(tmp.sos_adj_margin_last10_diff, bins=10)
tmp.groupby('sos_bin', observed=True).home_win.mean().plot(ax=axes[1], marker='o', color='darkorange')
axes[1].axhline(0.5, color='gray', linestyle='--')
axes[1].set_title('Heimsieg-Rate je SoS-adjustierter Margin-Diff')
axes[1].set_xlabel('Diff (Heim - Auswaerts)')
axes[1].set_ylabel('Heimsieg-Rate')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 6. Modell-Vergleich: alle Feature-Sets

Drei Konfigurationen:
1. **Basis** — ELO + Form + H2H + Rest (Notebook 03)
2. **+ Player Box-Score** (Notebook 05)
3. **+ Star Availability + SoS** (dieses Notebook)

In [ ]:
BASIS = [
    'home_elo_pre', 'away_elo_pre', 'elo_diff',
    'h2h_home_winrate_last5',
    'home_win_rate_last_5', 'home_win_rate_last_10', 'home_win_rate_last_20',
    'away_win_rate_last_5', 'away_win_rate_last_10', 'away_win_rate_last_20',
    'win_rate_diff_5', 'win_rate_diff_10', 'win_rate_diff_20',
    'home_avg_margin_last_5', 'home_avg_margin_last_10', 'home_avg_margin_last_20',
    'away_avg_margin_last_5', 'away_avg_margin_last_10', 'away_avg_margin_last_20',
    'margin_diff_5', 'margin_diff_10', 'margin_diff_20',
    'home_days_since_last_game', 'away_days_since_last_game',
    'home_is_back_to_back', 'away_is_back_to_back', 'rest_diff',
]
PLAYER_BOX = [c for c in g.columns if any(c.endswith(s)
    for s in ['_roll10', '_roll10_diff'])]
ADVANCED = [
    'home_top5_avail_last3', 'away_top5_avail_last3', 'top5_avail_diff',
    'home_sos_last10', 'away_sos_last10', 'sos_last10_diff',
    'home_quality_win_rate_last10', 'away_quality_win_rate_last10', 'quality_win_rate_last10_diff',
    'home_sos_adj_margin_last10', 'away_sos_adj_margin_last10', 'sos_adj_margin_last10_diff',
]
print('Basis:', len(BASIS), '| Player-Box:', len(PLAYER_BOX), '| Advanced:', len(ADVANCED))

In [ ]:
def evaluate(features, name, df_):
    df_ = df_.dropna(subset=features + ['home_win']).copy()
    train = df_[df_.season < 2019]
    test = df_[df_.season >= 2019]
    model = xgb.XGBClassifier(
        n_estimators=400, max_depth=4, learning_rate=0.05,
        subsample=0.9, colsample_bytree=0.9,
        eval_metric='logloss', random_state=42, n_jobs=-1, verbosity=0,
    )
    model.fit(train[features], train['home_win'])
    proba = model.predict_proba(test[features])[:, 1]
    pred = (proba >= 0.5).astype(int)
    return model, {
        'name': name, 'n_features': len(features), 'n_train': len(train), 'n_test': len(test),
        'accuracy': accuracy_score(test['home_win'], pred),
        'log_loss': log_loss(test['home_win'], proba),
        'brier':    brier_score_loss(test['home_win'], proba),
        'auc':      roc_auc_score(test['home_win'], proba),
    }

_, r1 = evaluate(BASIS, 'Basis (ELO + Form)', g)
_, r2 = evaluate(BASIS + PLAYER_BOX, '+ Player Box-Score', g)
model_full, r3 = evaluate(BASIS + PLAYER_BOX + ADVANCED, '+ Star Avail + SoS (FULL)', g)

comp = pd.DataFrame([r1, r2, r3]).set_index('name').round(4)
comp

## 7. Feature Importance — wo zieht es jetzt?

In [ ]:
all_feats = BASIS + PLAYER_BOX + ADVANCED
imp = pd.DataFrame({
    'feature': all_feats,
    'importance': model_full.feature_importances_,
    'group': ['Basis']*len(BASIS) + ['Player-Box']*len(PLAYER_BOX) + ['Advanced']*len(ADVANCED),
}).sort_values('importance', ascending=False)

fig, ax = plt.subplots(figsize=(8, 14))
color_map = {'Basis': 'steelblue', 'Player-Box': 'darkorange', 'Advanced': 'crimson'}
colors = [color_map[g] for g in imp.group]
ax.barh(imp.feature, imp.importance, color=colors)
ax.set_title('Feature Importance (rot = neu in Notebook 06)')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

print('\nMittlere Importance pro Feature-Gruppe:')
print(imp.groupby('group').importance.mean().round(4))

## 8. Aussicht

Wir haben jetzt drei Feature-Gruppen verglichen. Die naechste Stufe — **die wirklich interessant ist fuer 'wer wird Champion?':**

**Notebook 07 — Series-Simulation (Monte Carlo):**
- Aus Game-Win-Probabilities wird eine Best-of-7 simuliert (10.000 Mal pro Series)
- Aus Series-Wahrscheinlichkeiten wird ein ganzer Playoff-Bracket simuliert
- Jedes Team bekommt eine Championship-Wahrscheinlichkeit
- Historischer Backtest: Hatten wir den tatsaechlichen Champion in den Top-5?